# **G2P Evaluation on Wiktionary Phonemic Data**

This notebook trains and evaluates the Phonetisaurus G2P model used by the `tagalog_g2p.py`
command-line tool. It learns from the cleaned Wiktionary data in
`data/extra data/clean_tgl_wik.csv`, following the Phase 2 method.

Only the phonemic transcription is used. The words are split 80/10/10 so that training covers
every grapheme and phoneme, because Phonetisaurus silently drops a test word with a grapheme it
never saw in training. The model uses the default settings with only `--seq2_del` turned on, and
is scored by phone error rate (PER).

This notebook also sets up the comparison of G2P candidates. The test split, validation split and
10 cross-validation folds it writes are shared by every candidate: the rule-based Epitran
(`Epitran_eval.ipynb`) and the neural `filipino-byt5-g2p` (`ByT5_eval.ipynb`) and fine-tuned
CharsiuG2P (`ByT5_finetune.ipynb`). `G2P_compare.ipynb` compares them all with significance tests
and chooses the G2P, so run this notebook first.

Run the notebook inside the Docker container (`./run_phonetisaurus.ps1`), where Phonetisaurus is
installed. The split dictionaries are saved to `dicts/`, the model to `train/cwik_model.fst` and
the test predictions to `output/cwik_output.txt`. Scratch files (the test word list, training
by-products) live in a temporary directory that the last cell deletes.

**The process at a glance**

1. **Load** the CSV, keep the phonemic transcriptions and drop the broken ones.
2. **Split** the words 80/10/10, making sure training covers every symbol.
3. **Write** the split dictionaries.
4. **Train** the final model on the training and validation splits combined (90%).
5. **Cross-validate** with 10 folds to check how stable the PER is.
6. **Test** the model once on the held-out test split.
7. **Analyze** the errors, including where the glottal stop goes wrong and vowel-height
   confusions.
8. **Compare** the PER with published Tagalog G2P results.
9. **Clean up** the temporary files.

In [1]:
import subprocess
import sys
import os
import random
import tempfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

sys.dont_write_bytecode = True  # no __pycache__ in ../utils
sys.path.append(os.path.abspath("../utils"))
from tsv_metrics import tokenize_ipa, compute_per_wer, collect_errors
from train_tools import train_model

current_dir = Path().resolve()
data_csv    = current_dir.parent / "data" / "extra data" / "clean_tgl_wik.csv"
dicts_dir   = current_dir / "dicts"
train_dir   = current_dir / "train"
output_dir  = current_dir / "output"

for d in (dicts_dir, output_dir):
    d.mkdir(exist_ok=True)

# Scratch files (test word list, training by-products) live here; the directory is
# deleted by the Cleanup cell at the end (or when the kernel exits).
tmp = tempfile.TemporaryDirectory(prefix="wik_eval_")
tmp_dir = Path(tmp.name)

print("Working directory:", current_dir)
print("Source CSV       :", data_csv)

Working directory: /workspaces/Taglog-G2P/notebook
Source CSV       : /workspaces/Taglog-G2P/data/extra data/clean_tgl_wik.csv


## 1. Load the dictionary

The CSV has a `word` column and two transcriptions, `phonemic` and `phonetic`. Phase 2 uses the
phonemic one only, so `phonetic` is dropped.

The file is UTF-8 apart from one stray latin-1 byte, so it is read with
`encoding_errors="replace"`. `tokenize_ipa()` then splits each transcription into phones. It drops
stress and syllable marks (`ˈ ˌ .`), and keeps length marks, combining diacritics and tie-bar
affricates attached to their phone. Multi-word entries are skipped, since Phonetisaurus works one
word at a time.

Two checks drop transcriptions that are errors in the source data:

- **Outside the phoneme inventory.** A transcription with a symbol outside the Filipino phoneme
  inventory of Marqueses et al. (2026, Table 1) is dropped. This removes **4** entries: three
  with tone numbers (`himbing` as /him³³bin²⁴/) and `hiyas` as /ʃoʝas/.
- **Truncated.** A few transcriptions are only a fragment of the word (`insenso` as /sj/,
  `panggasi` as /ŋɡ/). They have at most 0.29 phones per letter, while the lowest real entries
  (`'ng` as /ŋ/) have 0.5, so an entry with fewer than 0.4 phones per letter is dropped. This
  removes **7** entries.

Of the **21,721** rows, **21,710** single-word entries remain.

In [2]:
df = pd.read_csv(data_csv, encoding="utf-8", encoding_errors="replace")
print("Raw rows:", len(df))
print("Columns :", list(df.columns))

# Keep phonemic only; drop the phonetic column entirely.
df = df[["word", "phonemic"]].copy()

# Filipino phoneme inventory (Marqueses et al., 2026, Table 1). A transcription with any other
# symbol, such as a tone number, is an error in the source data.
INVENTORY = {
    "p", "b", "k", "ɡ", "t", "d", "ʔ", "m", "ŋ", "n", "ɾ", "f", "v", "s", "z", "ʃ", "ʒ", "h",
    "t͡s", "d͡z", "t͡ʃ", "d͡ʒ", "w", "j", "l", "e", "i", "a", "ə", "o", "u",
}

entries = []          # list of (word, [phones])
skipped_multiword = 0
skipped_empty = 0
skipped_inventory = []
skipped_truncated = []

for word, phonemic in zip(df["word"].astype(str), df["phonemic"].astype(str)):
    word = word.strip()
    phonemic = phonemic.strip()
    if not word or not phonemic or phonemic == "nan":
        skipped_empty += 1
        continue
    if " " in word:                    # Phonetisaurus is word-by-word
        skipped_multiword += 1
        continue
    phones = tokenize_ipa(phonemic)
    if not phones:
        skipped_empty += 1
        continue
    if not set(phones) <= INVENTORY:
        skipped_inventory.append(word)
        continue
    # A few transcriptions are fragments of the word (insenso -> sj). They have at most 0.29 phones
    # per letter; the next lowest real entries ('ng -> ŋ) have 0.5.
    if len(phones) < 0.4 * sum(ch.isalpha() for ch in word):
        skipped_truncated.append(word)
        continue
    entries.append((word, phones))

print(f"Usable single-word entries : {len(entries)}")
print(f"Multi-word skipped         : {skipped_multiword}")
print(f"Empty/untokenizable skipped: {skipped_empty}")
print(f"Outside the inventory      : {len(skipped_inventory)} {skipped_inventory}")
print(f"Truncated transcriptions   : {len(skipped_truncated)} {skipped_truncated}")

Raw rows: 21721
Columns : ['word', 'phonemic', 'phonetic']
Usable single-word entries : 21710
Multi-word skipped         : 0
Empty/untokenizable skipped: 0
Outside the inventory      : 4 ['himbing', 'hiyas', 'kuwekong', 'liyempo']
Truncated transcriptions   : 7 ['apresasyon', 'asosasyon', 'barangay', 'depresasyon', 'dyahe', 'insenso', 'panggasi']


## 2. Split the data without unseen graphemes

A plain random split could put every word with a rare grapheme (`q`, `x`, `z`) into validation or
test. Phonetisaurus would then have no arc for that grapheme and would silently drop the word,
which corrupts the evaluation.

So the split is seeded for coverage. Going through the graphemes and then the phonemes from
rarest to most common, one word containing each symbol not yet covered goes into training. Every
grapheme then appears in training, so no validation or test word can contain an unseen one. A
final sweep checks this and moves any offending word back to training.

The split gives **17,368** training, **2,171** validation and **2,171** test words. **15** seed
words cover all 31 graphemes and 28 phonemes, and the sweep moves nothing.

In [3]:
random.seed(42)
shuffled = entries[:]
random.shuffle(shuffled)

# Global grapheme / phoneme frequency (rarest first drives coverage seeding).
graph_freq = Counter()
phone_freq = Counter()
for word, phones in shuffled:
    graph_freq.update(set(word))
    phone_freq.update(set(phones))


def coverage_seed(shuffled, graph_freq, phone_freq):
    """Pick a minimal set of word indices so train covers every symbol.

    Symbols are covered rarest-first; for each still-uncovered grapheme/phoneme
    we grab one word that contains it. Returns a set of indices into `shuffled`.
    """
    covered_g, covered_p = set(), set()
    seed_idx = set()

    def first_word_with(pred):
        for i, (word, phones) in enumerate(shuffled):
            if i in seed_idx:
                continue
            if pred(word, phones):
                return i
        return None

    for g, _ in sorted(graph_freq.items(), key=lambda kv: kv[1]):
        if g in covered_g:
            continue
        i = first_word_with(lambda w, p, g=g: g in w)
        if i is not None:
            seed_idx.add(i)
            covered_g.update(shuffled[i][0])
            covered_p.update(shuffled[i][1])

    for ph, _ in sorted(phone_freq.items(), key=lambda kv: kv[1]):
        if ph in covered_p:
            continue
        i = first_word_with(lambda w, p, ph=ph: ph in p)
        if i is not None:
            seed_idx.add(i)
            covered_g.update(shuffled[i][0])
            covered_p.update(shuffled[i][1])

    return seed_idx


seed_idx = coverage_seed(shuffled, graph_freq, phone_freq)

n = len(shuffled)
n_val  = n // 10
n_test = n // 10

# Seed words go to train; val/test are carved only from non-seed words.
rest = [e for i, e in enumerate(shuffled) if i not in seed_idx]
seed_entries = [shuffled[i] for i in seed_idx]

test  = rest[:n_test]
val   = rest[n_test:n_test + n_val]
train = seed_entries + rest[n_test + n_val:]


def train_graphemes(train):
    g = set()
    for word, _ in train:
        g.update(word)
    return g


# Safety sweep: move any val/test word with an OOV grapheme back into train.
moved = 0
changed = True
while changed:
    changed = False
    tg = train_graphemes(train)
    for split in (val, test):
        keep = []
        for word, phones in split:
            if set(word) <= tg:
                keep.append((word, phones))
            else:
                train.append((word, phones))
                moved += 1
                changed = True
        split[:] = keep

# Verify no OOV graphemes remain in val/test.
tg = train_graphemes(train)
oov_val  = [w for w, _ in val  if not set(w) <= tg]
oov_test = [w for w, _ in test if not set(w) <= tg]
assert not oov_val and not oov_test, f"OOV remains: {oov_val[:5]} {oov_test[:5]}"

# Coverage report.
all_g, all_p = set(graph_freq), set(phone_freq)
tp = set()
for _, phones in train:
    tp.update(phones)

print(f"Split sizes -> train: {len(train)}  val: {len(val)}  test: {len(test)}"
      f"  (total {len(train)+len(val)+len(test)})")
print(f"Coverage seed words        : {len(seed_idx)}")
print(f"Words moved by safety sweep: {moved}")
print(f"Grapheme coverage in train : {len(tg & all_g)}/{len(all_g)}")
print(f"Phoneme  coverage in train : {len(tp & all_p)}/{len(all_p)}")
assert tg >= all_g, "train is missing some grapheme"
assert tp >= all_p, "train is missing some phoneme"

Split sizes -> train: 17368  val: 2171  test: 2171  (total 21710)
Coverage seed words        : 15
Words moved by safety sweep: 0
Grapheme coverage in train : 31/31
Phoneme  coverage in train : 28/28


## 3. Write the dictionaries

Phonetisaurus reads one entry per line: the word, a tab, then the phones separated by spaces. The
files use a `cwik_` prefix so they don't overwrite the `tsv_*` and `train.*` files of the other
notebooks.

| File | Entries | Used for |
| --- | --- | --- |
| `dicts/cwik_train.dict` | 17,368 | training split |
| `dicts/cwik_val.dict` | 2,171 | validation split |
| `dicts/cwik_trainval.dict` | 19,539 | cross-validation and the final model |
| `dicts/cwik_test.tsv` | 2,171 unique words | gold reference for the test |

The test words also go to a temporary word list for the model.

In [4]:
def write_dict(entries, path):
    with open(path, "w", encoding="utf-8") as f:
        for word, phones in entries:
            f.write(f"{word}\t{' '.join(phones)}\n")


trainval = train + val

cwik_train_dict    = dicts_dir / "cwik_train.dict"
cwik_val_dict      = dicts_dir / "cwik_val.dict"
cwik_trainval_dict = dicts_dir / "cwik_trainval.dict"
cwik_test_tsv      = dicts_dir / "cwik_test.tsv"
cwik_test_wlist    = tmp_dir / "cwik_test.wlist"

write_dict(train,    cwik_train_dict)
write_dict(val,      cwik_val_dict)
write_dict(trainval, cwik_trainval_dict)
write_dict(test,     cwik_test_tsv)

with open(cwik_test_wlist, "w", encoding="utf-8") as f:
    for word, _ in test:
        f.write(word + "\n")

# Gold reference for scoring (reuse the already-tokenized phones).
ref = {word: phones for word, phones in test}

print(f"train.dict    : {len(train)} entries -> {cwik_train_dict.name}")
print(f"val.dict      : {len(val)} entries -> {cwik_val_dict.name}")
print(f"trainval.dict : {len(trainval)} entries (90% CV / final-train pool)")
print(f"test refs     : {len(ref)} unique words -> {cwik_test_tsv.name}")

train.dict    : 17368 entries -> cwik_train.dict
val.dict      : 2171 entries -> cwik_val.dict
trainval.dict : 19539 entries (90% CV / final-train pool)
test refs     : 2171 unique words -> cwik_test.tsv


## 4. Train the model

Default Phonetisaurus settings with only `--seq2_del` turned on, which lets a grapheme map to no
phoneme (so the digraph *ng* can become a single /ŋ/). These are the baseline settings of Aquino et
al. (2019), fixed before this notebook was run. The check below confirms the choice on the
validation split, so the test split isn't used to choose anything. The final model is then
trained on the combined training and validation pool (90%).

It is saved as `train/cwik_model.fst`, the file the command-line tool loads, so rerunning this
cell replaces the tool's model.

### Check the settings on the validation split

Three settings are trained on the training split and scored on the validation split:

| Settings | Source |
| --- | --- |
| default + `--seq2_del` | the baseline of Aquino et al. (2019), used here |
| default | Phonetisaurus without any options |
| `ngram_order` 4, `seq1_max` 1, `seq2_max` 2, `--seq2_del`, `--grow` | the best settings of the grid search in `Aquino_eval.ipynb` |

`--seq2_del` gives the lowest validation PER, **0.96%** (WER 6.59%), against 1.03% without it and
1.05% for the grid-search settings, so it is kept for the final model.

In [5]:
val_ref = {word: phones for word, phones in val}
val_wlist = tmp_dir / "cwik_val.wlist"
val_wlist.write_text("\n".join(val_ref), encoding="utf-8")

settings = {
    "default + --seq2_del": ["--seq2_del"],
    "default": [],
    "Aquino_eval grid-search best": ["--ngram_order", "4", "--seq1_max", "1", "--seq2_max", "2",
                                     "--seq2_del", "--grow"],
}

rows = []
for i, (name, flags) in enumerate(settings.items()):
    train_model(cwik_train_dict, f"val_{i}", tmp_dir / "val_models", tmp_dir / "val_train", *flags)
    out = subprocess.run(
        ["phonetisaurus-apply", "--model", str(tmp_dir / "val_models" / f"val_{i}.fst"),
         "--word_list", str(val_wlist)],
        capture_output=True, text=True,
    ).stdout
    val_hyp = {p[0]: p[1].split() for p in (l.split("\t") for l in out.splitlines()) if len(p) >= 2}
    m = compute_per_wer(val_ref, val_hyp)
    rows.append({"Settings": name, "PER (%)": round(m["per"] * 100, 2),
                 "WER (%)": round(m["wer"] * 100, 2), "Words": m["n_words"]})

pd.DataFrame(rows)

,Settings,PER (%),WER (%),Words
0,default + --seq2_del,0.96,6.59,2171
1,default,1.03,7.00,2171
2,Aquino_eval grid-search best,1.05,7.32,2171


In [6]:
cwik_model = train_dir / "cwik_model"

result = train_model(cwik_trainval_dict, "cwik_model", train_dir, tmp_dir / "train", "--seq2_del")
print(result.stderr)

INFO:phonetisaurus-train:2026-10-02 19:03:45:  Checking command configuration...
INFO:phonetisaurus-train:2026-10-02 19:03:45:  Checking lexicon for reserved characters: '}', '|', '_'...
INFO:phonetisaurus-train:2026-10-02 19:03:45:  Aligning lexicon...
INFO:phonetisaurus-train:2026-10-02 19:04:08:  Training joint ngram model...
INFO:phonetisaurus-train:2026-10-02 19:04:08:  Converting ARPA format joint n-gram model to WFST format...
INFO:phonetisaurus-train:2026-10-02 19:04:10:  G2P training succeeded: /tmp/wik_eval_cvsqgs1t/train/cwik_model.fst



## 5. Cross-validation

10-fold cross-validation over the same 90% pool. Each round trains on nine folds and tests on the
tenth, in its own temporary directory. This estimates how stable the PER is. It doesn't pick a
model.

The fold PERs are saved to `output/cwik_cv.tsv` for `G2P_compare.ipynb`. The mean PER across the
folds is **0.99%** (± 0.06%).

In [7]:
random.seed(42)
cv_entries = trainval[:]
random.shuffle(cv_entries)

N_FOLDS = 10
fold_size = len(cv_entries) // N_FOLDS
folds = [cv_entries[i * fold_size:(i + 1) * fold_size] for i in range(N_FOLDS)]
folds[-1] += cv_entries[N_FOLDS * fold_size:]   # remainder into last fold
print("Fold sizes:", [len(f) for f in folds])

fold_pers = []
for k in range(N_FOLDS):
    held_out   = folds[k]
    train_fold = [e for i, f in enumerate(folds) if i != k for e in f]

    with tempfile.TemporaryDirectory() as fold_tmp:
        fold_tmp = Path(fold_tmp)
        fold_dict  = fold_tmp / "fold.dict"
        fold_model = fold_tmp / "fold"
        fold_wlist = fold_tmp / "held.wlist"
        fold_out   = fold_tmp / "held.out"

        write_dict(train_fold, fold_dict)

        seen, wl = set(), []
        for word, _ in held_out:
            if word not in seen:
                seen.add(word)
                wl.append(word)
        fold_wlist.write_text("\n".join(wl), encoding="utf-8")

        subprocess.run(
            ["phonetisaurus-train", "--lexicon", str(fold_dict),
             "--seq2_del", "--dir_prefix", str(fold_tmp), "--model_prefix", fold_model.name],
            capture_output=True, text=True,
        )
        with open(fold_out, "w", encoding="utf-8") as f:
            subprocess.run(
                ["phonetisaurus-apply", "--model", str(fold_model) + ".fst",
                 "--word_list", str(fold_wlist)],
                stdout=f, text=True,
            )

        hyp = {}
        with open(fold_out, encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split("\t")
                if len(parts) >= 2:
                    hyp[parts[0]] = parts[1].split()

        fold_ref = {w: p for w, p in held_out}
        m = compute_per_wer(fold_ref, hyp)
        fold_pers.append(m["per"])
        print(f"  Fold {k + 1:2d}: PER {m['per'] * 100:5.2f}%  ({m['n_words']} words)")

fold_pers = np.array(fold_pers)

# Saved for G2P_compare.ipynb.
pd.DataFrame({"fold": range(1, N_FOLDS + 1), "per": fold_pers}).to_csv(
    output_dir / "cwik_cv.tsv", sep="\t", index=False)
print()
print(f"10-fold CV PER: mean {fold_pers.mean() * 100:.2f}%  +/-  {fold_pers.std() * 100:.2f}%")

Fold sizes: [1953, 1953, 1953, 1953, 1953, 1953, 1953, 1953, 1953, 1962]


  Fold  1: PER  0.96%  (1953 words)


  Fold  2: PER  0.96%  (1953 words)


  Fold  3: PER  0.97%  (1953 words)


  Fold  4: PER  1.09%  (1953 words)


...

  Fold  6: PER  0.92%  (1953 words)


  Fold  7: PER  1.00%  (1953 words)


  Fold  8: PER  1.11%  (1953 words)


  Fold  9: PER  0.99%  (1953 words)


  Fold 10: PER  0.95%  (1962 words)

10-fold CV PER: mean 0.99%  +/-  0.06%


## 6. Test the final model

The final model is applied once to the held-out test split. PER is the main metric (the only one
in Phase 2), and word error rate (WER) is reported second. An `assert` checks that every test word
got a prediction, which confirms the split prevented any unseen-grapheme drops.

Predictions are saved to `output/cwik_output.txt`. The model reaches a PER of **0.81%** and a WER
of **5.85%**: 2,044 of the 2,171 test words are fully correct.

In [8]:
cwik_output = output_dir / "cwik_output.txt"

with open(cwik_output, "w", encoding="utf-8") as f:
    subprocess.run(
        ["phonetisaurus-apply", "--model", str(cwik_model) + ".fst",
         "--word_list", str(cwik_test_wlist)],
        stdout=f, text=True,
    )

hyp = {}
with open(cwik_output, encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            hyp[parts[0]] = parts[1].split()

metrics = compute_per_wer(ref, hyp)

# No test word should be dropped for OOV.
assert metrics["n_words"] == len(ref), (
    f"OOV drop: scored {metrics['n_words']} of {len(ref)} test words"
)

print("=== Final Model (trainval.dict, default + --seq2_del) on held-out test ===")
print(f"  PER : {metrics['per']:.4f}  ({metrics['per'] * 100:.2f}%)   <- primary metric")
print(f"  WER : {metrics['wer']:.4f}  ({metrics['wer'] * 100:.2f}%)")
print(f"  Words evaluated  : {metrics['n_words']}")
print(f"  Correct words    : {metrics['n_correct']}")
print(f"  Total ref phones : {metrics['n_ref_phones']}")
print(f"  Total edit dist  : {metrics['n_edit']}")

=== Final Model (trainval.dict, default + --seq2_del) on held-out test ===
  PER : 0.0081  (0.81%)   <- primary metric
  WER : 0.0585  (5.85%)
  Words evaluated  : 2171
  Correct words    : 2044
  Total ref phones : 17303
  Total edit dist  : 141


## 7. Error analysis

Phase 2 asks for more than an overall PER. It looks for critical phonetic shifts, errors that
could make the downstream ALINE algorithm misjudge a drug-name pair. The errors are broken down by
type, by phone and by substitution pair, with example words.

### 7.1 Error types

There are **141** edits in total: 13 substitutions, 56 deletions and 72 insertions.

In [9]:
err = collect_errors(ref, hyp)
total_errors = err["n_sub"] + err["n_del"] + err["n_ins"]

error_div = pd.DataFrame([
    {"Type": "Substitution", "Count": err["n_sub"], "% of Errors": f"{err['n_sub'] / total_errors * 100:.1f}%"},
    {"Type": "Deletion",     "Count": err["n_del"], "% of Errors": f"{err['n_del'] / total_errors * 100:.1f}%"},
    {"Type": "Insertion",    "Count": err["n_ins"], "% of Errors": f"{err['n_ins'] / total_errors * 100:.1f}%"},
    {"Type": "TOTAL",        "Count": total_errors, "% of Errors": "100.0%"},
])
print(f"Total edit operations: {total_errors}")
error_div

Total edit operations: 141


,Type,Count,% of Errors
0,Substitution,13,9.2%
1,Deletion,56,39.7%
2,Insertion,72,51.1%
3,TOTAL,141,100.0%


### 7.2 Phones behind each error type

Almost every deletion and insertion is the glottal stop `ʔ`, which is dropped **50** times and
added **65** times.

In [10]:
def top10(counter, label):
    print(f"--- {label} ---")
    return pd.DataFrame([{"Phone": p, "Count": c} for p, c in counter.most_common(10)])

print("Ref phones most often substituted:")
display(top10(err["sub_ref_counter"], "REF phones replaced"))
print("Ref phones most often deleted:")
display(top10(err["del_counter"], "REF phones dropped"))
print("Phones most often inserted:")
display(top10(err["ins_counter"], "phones added"))

Ref phones most often substituted:
--- REF phones replaced ---


,Phone,Count
0,j,5
1,i,1
2,ʔ,1
3,w,1
4,d͡ʒ,1
5,u,1
6,s,1
7,ŋ,1
8,t͡ʃ,1


Ref phones most often deleted:
--- REF phones dropped ---


,Phone,Count
0,ʔ,50
1,i,1
2,ɡ,1
3,n,1
4,a,1
5,ɾ,1
6,w,1


Phones most often inserted:
--- phones added ---


,Phone,Count
0,ʔ,65
1,u,2
2,d,1
3,i,1
4,h,1
5,w,1
6,t,1


### 7.3 Substitution pairs

Substitutions are rare. The most common one, `j → i`, happens 3 times.

In [11]:
pair_rows = [
    {"Pair": f"{r} -> {h}", "Count": c}
    for (r, h), c in err["sub_pair_counter"].most_common(10)
]
display(pd.DataFrame(pair_rows))

print("\nExample words for the top substitution pairs (critical phonetic shifts):")
for (r, h), c in err["sub_pair_counter"].most_common(5):
    print(f"\n  {r} -> {h}  (count={c})")
    for ex in err["sub_pair_examples"].get((r, h), [])[:3]:
        print(f"    {ex['word']:<20}  ref:[{ex['ref']}]  hyp:[{ex['hyp']}]")

,Pair,Count
0,j -> i,3
1,i -> j,1
2,ʔ -> j,1
3,w -> ʔ,1
4,d͡ʒ -> s,1
5,j -> ʔ,1
6,u -> w,1
7,s -> ʔ,1
8,j -> ʃ,1
9,ŋ -> n,1



Example words for the top substitution pairs (critical phonetic shifts):

  j -> i  (count=3)
    alkaide               ref:[ʔ a l k a j d e]  hyp:[ʔ a l k a ʔ i d e]
    plais                 ref:[p l a j s]  hyp:[p l a ʔ i s]
    geim                  ref:[ɡ e j m]  hyp:[ɡ e ʔ i m]

  i -> j  (count=1)
    panainga              ref:[p a n a ʔ i ŋ a]  hyp:[p a n a j ŋ a]

  ʔ -> j  (count=1)
    kabaian               ref:[k a b a ʔ i ʔ a n]  hyp:[k a b a j a n]

  w -> ʔ  (count=1)
    mamituin              ref:[m a m i t w i n]  hyp:[m a m i t u ʔ i n]

  d͡ʒ -> s  (count=1)
    masads                ref:[m a s a d͡ʒ]  hyp:[m a s a d s]


### 7.4 Deletion and insertion examples

Example words for the most often deleted and most often inserted phones.

In [12]:
print("Example words for the top deleted ref phones (dropped in hyp):")
for r, c in err["del_counter"].most_common(5):
    print(f"\n  del({r})  (count={c})")
    for ex in err["del_examples"].get(r, [])[:3]:
        print(f"    {ex['word']:<20}  ref:[{ex['ref']}]  hyp:[{ex['hyp']}]")

print("\nExample words for the top inserted phones (added in hyp):")
for h, c in err["ins_counter"].most_common(5):
    print(f"\n  ins({h})  (count={c})")
    for ex in err["ins_examples"].get(h, [])[:3]:
        print(f"    {ex['word']:<20}  ref:[{ex['ref']}]  hyp:[{ex['hyp']}]")

Example words for the top deleted ref phones (dropped in hyp):

  del(ʔ)  (count=50)
    panainga              ref:[p a n a ʔ i ŋ a]  hyp:[p a n a j ŋ a]
    kato                  ref:[k a t o ʔ]  hyp:[k a t o]
    umiiyak               ref:[ʔ u m i ʔ i a k]  hyp:[ʔ u m i i a k]

  del(i)  (count=1)
    kabaian               ref:[k a b a ʔ i ʔ a n]  hyp:[k a b a j a n]

  del(ɡ)  (count=1)
    mongol                ref:[m o ŋ ɡ o l]  hyp:[m o ŋ o l]

  del(n)  (count=1)
    ng                    ref:[n a ŋ]  hyp:[ŋ]

  del(a)  (count=1)
    ng                    ref:[n a ŋ]  hyp:[ŋ]

Example words for the top inserted phones (added in hyp):

  ins(ʔ)  (count=65)
    lider-kabataan        ref:[l i d e ɾ k a b a t a ʔ a n]  hyp:[l i d e ɾ ʔ k a b a t a ʔ a n]
    albahaka              ref:[ʔ a l b a h a k a]  hyp:[ʔ a l b a h a k a ʔ]
    alhebraika            ref:[ʔ a l h e b ɾ a j k a]  hyp:[ʔ a l h e b ɾ a j k a ʔ]

  ins(u)  (count=2)
    mamituin              ref:[m a m i t w i n]  

### 7.5 Where the glottal stop goes wrong

Counts alone don't say where in the word `ʔ` fails. So each `ʔ` deletion (by its position in the
reference) and each `ʔ` insertion (by its position in the prediction) is put into one of three
buckets, **initial**, **medial** or **final**, using the alignment from `align_trxn()`.

Nearly all of them are word-final: **88.0%** of the deletions and **87.7%** of the insertions.
This comes from the language, not the model. Tagalog spelling doesn't mark a word-final glottal
stop, so `kato` (/katoʔ/) and `kamisa` (/kamisa/) look the same to the model. It drops a final `ʔ`
where one belongs and adds one where it doesn't, and spelling alone can't settle either case.

In [13]:
from alignment import align_trxn

GLOTTAL = "ʔ"
_MAX_POS_EXAMPLES = 8


def _pos_bucket(idx, n):
    """Classify a phone index within a word of length n."""
    if idx == 0:
        return "initial"
    if idx == n - 1:
        return "final"
    return "medial"


# Walk each alignment, tracking the true index within the ref/hyp words, and
# bucket every glottal-stop deletion (by ref position) and insertion (by hyp
# position) into initial / medial / final. Word-final cases are also collected
# as concrete examples.
del_pos = Counter()
ins_pos = Counter()
del_final_examples = []   # (word, ref_str, hyp_str) — real final ʔ dropped
ins_final_examples = []   # (word, ref_str, hyp_str) — bogus final ʔ added

for word, ref_phones in ref.items():
    if word not in hyp:
        continue
    hyp_phones = hyp[word]
    alnA, alnB, _, _ = align_trxn(ref_phones, hyp_phones)
    ref_str, hyp_str = " ".join(ref_phones), " ".join(hyp_phones)

    ri = hi = 0
    for a, b in zip(alnA, alnB):
        if a == GLOTTAL and b == "*":          # deletion: ref has ʔ, hyp dropped it
            pos = _pos_bucket(ri, len(ref_phones))
            del_pos[pos] += 1
            if pos == "final" and len(del_final_examples) < _MAX_POS_EXAMPLES:
                del_final_examples.append((word, ref_str, hyp_str))
        if b == GLOTTAL and a == "*":          # insertion: hyp added a spurious ʔ
            pos = _pos_bucket(hi, len(hyp_phones))
            ins_pos[pos] += 1
            if pos == "final" and len(ins_final_examples) < _MAX_POS_EXAMPLES:
                ins_final_examples.append((word, ref_str, hyp_str))
        if a != "*":
            ri += 1
        if b != "*":
            hi += 1


def _pos_table(counter, label):
    total = sum(counter.values())
    rows = []
    for pos in ("initial", "medial", "final"):
        c = counter[pos]
        pct = f"{c / total * 100:.1f}%" if total else "-"
        rows.append({"Position": pos, "Count": c, "% of ʔ errors": pct})
    rows.append({"Position": "TOTAL", "Count": total, "% of ʔ errors": "100.0%"})
    print(f"--- {label} ---")
    return pd.DataFrame(rows)


print("Where in the word is the glottal stop ʔ mispredicted?\n")
display(_pos_table(del_pos, "ʔ DELETIONS  (ref has ʔ, model dropped it)"))
display(_pos_table(ins_pos, "ʔ INSERTIONS (model added a spurious ʔ)"))

# Concrete word-final examples: every one is a vowel-final spelling, so the same
# surface cue maps to opposite correct answers — unsolvable from orthography alone.
print("\nWord-final ʔ DELETIONS (a real final ʔ was dropped):")
for w, r, h in del_final_examples:
    print(f"    {w:16s}  ref:[{r}]   hyp:[{h}]")

print("\nWord-final ʔ INSERTIONS (a bogus final ʔ was added):")
for w, r, h in ins_final_examples:
    print(f"    {w:16s}  ref:[{r}]   hyp:[{h}]")


Where in the word is the glottal stop ʔ mispredicted?

--- ʔ DELETIONS  (ref has ʔ, model dropped it) ---


,Position,Count,% of ʔ errors
0,initial,1,2.0%
1,medial,5,10.0%
2,final,44,88.0%
3,TOTAL,50,100.0%


--- ʔ INSERTIONS (model added a spurious ʔ) ---


,Position,Count,% of ʔ errors
0,initial,0,0.0%
1,medial,8,12.3%
2,final,57,87.7%
3,TOTAL,65,100.0%



Word-final ʔ DELETIONS (a real final ʔ was dropped):
    kato              ref:[k a t o ʔ]   hyp:[k a t o]
    salanta           ref:[s a l a n t a ʔ]   hyp:[s a l a n t a]
    luga              ref:[l u ɡ a ʔ]   hyp:[l u ɡ a]
    bana              ref:[b a n a ʔ]   hyp:[b a n a]
    aba               ref:[ʔ a b a ʔ]   hyp:[ʔ a b a]
    salasa            ref:[s a l a s a ʔ]   hyp:[s a l a s a]
    sipa              ref:[s i p a ʔ]   hyp:[s i p a]
    mamamana          ref:[m a m a m a n a ʔ]   hyp:[m a m a m a n a]

Word-final ʔ INSERTIONS (a bogus final ʔ was added):
    albahaka          ref:[ʔ a l b a h a k a]   hyp:[ʔ a l b a h a k a ʔ]
    alhebraika        ref:[ʔ a l h e b ɾ a j k a]   hyp:[ʔ a l h e b ɾ a j k a ʔ]
    muskulo           ref:[m u s k u l o]   hyp:[m u s k u l o ʔ]
    tangke            ref:[t a ŋ k e]   hyp:[t a ŋ k e ʔ]
    kalbo             ref:[k a l b o]   hyp:[k a l b o ʔ]
    aribay-baho       ref:[ʔ a ɾ i b a j b a h o]   hyp:[ʔ a ɾ i b a j b a h o ʔ]
    

### 7.6 Vowel-height confusion: o↔u and e↔i

Tagalog *o*/*u* and *e*/*i* were historically allophones, so the model could swap them with no cue
from the spelling. Unlike a length or diacritic slip, such a swap changes a phoneme and can mislead
the downstream ALINE drug-name matcher. `vowel_confusion()` counts every o↔u and e↔i
substitution, in both directions, with example words.

There are **none**: the phonemic model keeps these vowels apart.

In [14]:
from tsv_metrics import vowel_confusion

vc = vowel_confusion(ref, hyp)   # default groups: o/u and e/i

display(pd.DataFrame([
    {"Confusion": k, "Count": v["count"], "% of subs": f"{v['rate'] * 100:.1f}%"}
    for k, v in vc.items()
]))

for label, v in vc.items():
    print(f"\n{label}: {v['count']} confusions")
    for (r, h), c in v["by_pair"].most_common():
        print(f"    {r} -> {h}: {c}")
    for ex in v["examples"][:5]:
        print(f"      {ex['word']:<18} ref:[{ex['ref']}]  hyp:[{ex['hyp']}]")

if all(v["count"] == 0 for v in vc.values()):
    print("\nNo o↔u or e↔i substitutions: the phonemic model keeps these vowels distinct.")

,Confusion,Count,% of subs
0,o/u,0,0.0%
1,e/i,0,0.0%



o/u: 0 confusions

e/i: 0 confusions

No o↔u or e↔i substitutions: the phonemic model keeps these vowels distinct.


## 8. Comparison

The test PER next to published Tagalog G2P results. These were measured on different datasets,
so the comparison is only a rough guide. The controlled comparison, with every candidate on this
test split, is in `G2P_compare.ipynb`.

In [15]:
comparison = pd.DataFrame([
    {"System": "This model (Wiktionary phonemic, default + seq2_del)",
     "Reported PER": f"{metrics['per'] * 100:.2f}%",
     "Dataset": "Wiktionary test split (this work)"},
    {"System": "Aquino et al. (2019) - monolingual statistical",
     "Reported PER": "5.87%",
     "Dataset": "Transcribed speaker corpus"},
    {"System": "Deri & Knight (2016) - cross-lingual WFST",
     "Reported PER": "5-10%",
     "Dataset": "Cross-lingual Wiktionary"},
])
comparison

,System,Reported PER,Dataset
0,"This model (Wiktionary phonemic, default + seq...",0.81%,Wiktionary test split (this work)
1,Aquino et al. (2019) - monolingual statistical,5.87%,Transcribed speaker corpus
2,Deri & Knight (2016) - cross-lingual WFST,5-10%,Cross-lingual Wiktionary


## 9. Clean up

Deletes the temporary directory with the test word list and training by-products. It is also
deleted automatically when the kernel shuts down.

In [16]:
tmp.cleanup()